# TriGuard — fine-tune DistilBERT for fake-job text (GPU)
Run on Colab with a GPU runtime (Runtime > Change runtime type > T4).
1. Upload `text_train.csv`, `text_val.csv`, `text_test.csv` (from `data/processed/`) when prompted.
2. Run all cells. Download `text_distilbert.zip` and unzip it into `models/text_distilbert/` in the repo.
The analyzer (`ml/text/analyzer.py`) picks it up automatically once `models/text_distilbert/meta.json` exists.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn
from google.colab import files
up = files.upload()   # text_train.csv, text_val.csv, text_test.csv

In [ ]:
import json, os, numpy as np, pandas as pd, torch
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression
from torch import nn
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          EarlyStoppingCallback, set_seed)
SEED, MAX_LEN = 42, 256
set_seed(SEED)
tr, va, te = (pd.read_csv(f"text_{s}.csv") for s in ("train", "val", "test"))
print(len(tr), len(va), len(te), tr.label.mean())

In [ ]:
class DS(torch.utils.data.Dataset):
    def __init__(self, df, tok):
        self.enc = tok(df.text.tolist(), truncation=True, max_length=MAX_LEN, padding="max_length")
        self.y = df.label.tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        d = {k: torch.tensor(v[i]) for k, v in self.enc.items()}; d["labels"] = torch.tensor(self.y[i]); return d

class WeightedTrainer(Trainer):
    """Cross-entropy with class weights to handle the scam/genuine imbalance."""
    def __init__(self, *a, class_weights=None, **k):
        super().__init__(*a, **k); self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=self.cw.to(out.logits.device))(out.logits, labels)
        return (loss, out) if return_outputs else loss

def metrics_fn(p):
    prob = torch.softmax(torch.tensor(p.predictions), -1)[:, 1].numpy()
    return {"pr_auc": average_precision_score(p.label_ids, prob), "roc_auc": roc_auc_score(p.label_ids, prob)}

def train(model_name, out_dir, epochs=6):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    pos = tr.label.mean(); cw = torch.tensor([1.0, (1 - pos) / pos], dtype=torch.float)
    args = TrainingArguments(out_dir, num_train_epochs=epochs, per_device_train_batch_size=16,
        per_device_eval_batch_size=64, learning_rate=3e-5, weight_decay=0.01, eval_strategy="epoch",
        save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="pr_auc",
        greater_is_better=True, save_total_limit=1, fp16=torch.cuda.is_available(), seed=SEED, report_to="none")
    trainer = WeightedTrainer(model=model, args=args, train_dataset=DS(tr, tok), eval_dataset=DS(va, tok),
        compute_metrics=metrics_fn, class_weights=cw, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()
    return trainer, tok

def probs(trainer, df, tok):
    logits = trainer.predict(DS(df, tok)).predictions
    return torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()

In [ ]:
trainer, tok = train("distilbert-base-uncased", "out_distilbert")
pv = probs(trainer, va, tok); pt = probs(trainer, te, tok)
# Platt calibration + F1 threshold, fitted on VALIDATION only
logit = lambda p: np.log(np.clip(p, 1e-6, 1-1e-6) / (1 - np.clip(p, 1e-6, 1-1e-6))).reshape(-1, 1)
cal = LogisticRegression(C=1e6, max_iter=1000).fit(logit(pv), va.label)
cv, ct = cal.predict_proba(logit(pv))[:, 1], cal.predict_proba(logit(pt))[:, 1]
grid = np.linspace(0.01, 0.99, 99)
thr = float(grid[int(np.argmax([f1_score(va.label, cv >= t) for t in grid]))])
yhat = ct >= thr
print(dict(threshold=thr, precision=precision_score(te.label, yhat), recall=recall_score(te.label, yhat),
           f1=f1_score(te.label, yhat), pr_auc=average_precision_score(te.label, ct),
           roc_auc=roc_auc_score(te.label, ct), mcc=matthews_corrcoef(te.label, yhat)))

In [ ]:
OUT = "text_distilbert"
trainer.save_model(OUT); tok.save_pretrained(OUT)
json.dump({"max_len": MAX_LEN, "threshold": thr, "platt_coef": float(cal.coef_[0][0]),
           "platt_intercept": float(cal.intercept_[0]), "base_model": "distilbert-base-uncased"},
          open(f"{OUT}/meta.json", "w"))
!zip -qr text_distilbert.zip text_distilbert
files.download("text_distilbert.zip")

## Optional: MuRIL (Indian-language, multilingual) variant
Same recipe with `google/muril-base-cased`. Unzip into `models/text_muril/` (not auto-selected by default; add it to `TEXT_MODEL_PRIORITY` in `ml/common/config.py` to use it).

In [ ]:
# trainer_m, tok_m = train("google/muril-base-cased", "out_muril")
# ...then repeat the calibration/save cells with OUT = "text_muril" and base_model "google/muril-base-cased".